# Run PASCHEN-1D

This notebook is the main user-facing driver for running a PASCHEN-1D simulation from one of the shipped configuration files, or from a custom configuration module with the same interface.

Typical workflow:

1. Choose `CONFIG_MODULE` in the first code cell.
2. Optionally add small case-specific overrides after `cfg = SimulationConfig()`.
3. Run the simulation. Outputs are written to the folder named by `cfg.run.run_name`.
4. Use the optional quick-look cell for a small immediate check, then use the dedicated diagnostics notebooks for full postprocessing.


## Select Configuration And Run

Choose one configuration module by uncommenting or editing `CONFIG_MODULE`. The loader imports that module as the active simulation configuration, then the notebook builds `cfg` and calls `run_simulation(cfg)`.

Use the override area only for small one-off changes such as `run_name`, voltage amplitude, grid size, or selected physical coefficients. For reusable cases, it is cleaner to create or edit a dedicated `config_case_*.py` file.


In [ ]:
# --------------------------------------------------------------------
# 0. Select configuration module
# --------------------------------------------------------------------
# Use one of the shipped config modules, or provide your own module name.
# A trailing .py suffix is allowed. The selected module must define
# SimulationConfig and SimulationState.
# CONFIG_MODULE = "config_case_argon_dc_discharge"
# CONFIG_MODULE = "config_case_nitrogen_pulsed_discharge"
# CONFIG_MODULE = "config_case_deuterium_pulsed_discharge"
# CONFIG_MODULE = "config_case_helium_photoemission_discharge"
CONFIG_MODULE = "config_case_argon_photoemission_discharge"
# CONFIG_MODULE = "config"  # optional generic config.py

from config_loader import load_simulation_case

SimulationConfig, run_simulation = load_simulation_case(CONFIG_MODULE)

# --------------------------------------------------------------------
# 1. Build configuration
# --------------------------------------------------------------------
# Optional case-specific overrides.
# Keep each run_name unique to avoid output-folder collisions.
# Example:
#   cfg = SimulationConfig()
#   cfg.run.run_name = "nitrogen_pulse_case_01"
#   cfg.waveform.V_peak = 130.0
cfg = SimulationConfig()

# --------------------------------------------------------------------
# 2. Run the simulation
# --------------------------------------------------------------------
state = run_simulation(cfg)
print(f"Saved run outputs in: {cfg.run.run_name}")


## Optional Quick-Look Plots

This cell reads the saved output from the run that just finished and plots a small user-selected set of quantities. It is meant for a fast sanity check, not for full analysis.

Set `QUICKLOOK_ENABLED = False` to skip it. Edit `quicklook_temporal` and `quicklook_spatial` to choose which quantities to plot. A single spatial quantity may be written as either `"ne"` or `("ne",)`.

For publication figures or detailed inspection, use `diagnostics_temporal_profiles.ipynb`, `diagnostics_spatial_snapshots.ipynb`, and `diagnostics_spatial_averages.ipynb`.


In [ ]:
QUICKLOOK_ENABLED = True

# Scalar histories to plot individually. Use names printed by
# print_available_diagnostics(ctx), e.g. "V_gap", "I_discharge", "cfl", "diffusion_cfl".
quicklook_temporal = (
    "cfl",
    "V_app",
    "V_gap",
#     "I_discharge"
)

# Spatial sampled quantities to plot individually. Use () to skip spatial plots.
# Single quantities may be written as "ne" or ("ne",).
# quicklook_spatial = ()
# Example:
# quicklook_spatial = ("ne", "ni", "S_ion")
quicklook_spatial = "ne"

# Plot controls.
quicklook_t_start = None
quicklook_t_end = None
quicklook_t_unit = "us"
quicklook_x_unit = "cm"
quicklook_spatial_t_samples = None  # None -> final snapshot; or tuple in seconds.
quicklook_spatial_yscale = "linear"
quicklook_save = False


def _as_quantity_tuple(quantities):
    """Normalize one quantity string or an iterable of quantity strings."""
    if quantities is None:
        return ()
    if isinstance(quantities, str):
        return (quantities,)
    return tuple(quantities)


quicklook_temporal = _as_quantity_tuple(quicklook_temporal)
quicklook_spatial = _as_quantity_tuple(quicklook_spatial)

if QUICKLOOK_ENABLED:
    from diagnostics_io import (
        load_run_context,
        print_available_diagnostics,
        set_notebook_plot_style,
        available_temporal,
        available_spatial,
        plot_temporal_quantity,
        plot_spatial_profiles,
    )

    ctx = load_run_context(cfg.run.run_name, ".")
    set_notebook_plot_style(font_size=11)

    # Uncomment once if you want to inspect all available names.
    # print_available_diagnostics(ctx)

    for quantity in quicklook_temporal:
        if quantity not in available_temporal(ctx):
            print(f"Quick-look temporal quantity unavailable: {quantity}")
            continue
        plot_temporal_quantity(
            ctx,
            quantity,
            t_start=quicklook_t_start,
            t_end=quicklook_t_end,
            t_unit=quicklook_t_unit,
            save=quicklook_save,
            fig_name=f"quicklook_{quantity}",
        )

    for quantity in quicklook_spatial:
        if quantity not in available_spatial(ctx):
            print(f"Quick-look spatial quantity unavailable: {quantity}")
            continue
        plot_spatial_profiles(
            ctx,
            quantity,
            t_samples=quicklook_spatial_t_samples,
            x_unit=quicklook_x_unit,
            time_unit_for_labels=quicklook_t_unit,
            yscale=quicklook_spatial_yscale,
            save=quicklook_save,
            fig_name=f"quicklook_{quantity}_spatial",
        )
